In [23]:
%load_ext autoreload
%autoreload 2

from nycschools import schools, exams, geo
from miximaps import ui, nyc
import geopandas as gpd
import pandas as pd
import folium

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
steam_coords = [40.698616076934954, -73.97122972391207]
# the dbn of the partner schools
dbns = [ "17K382", "13K595", "13K670", "16K455", "13K745", "17K600", "13K265", "13K605", "22K405", "17K590", "13K419", "17K543", "17K528", ]
df = schools.load_school_demographics()
points = geo.get_points()
feet = geo.load_school_footprints()
feet = geo.merge_campus_id(feet)
regents = exams.load_regents()
df = geo.merge_campus_id(df)
df = df[df.dbn.isin(dbns)]
df = df[df.ay == df.ay.max()]

loading: school-demographics.csv
loading: zipcodes.geojson
loading: school-building-footprints.geojson
loading: nyc-regents.csv


In [16]:
# merge some regents results into the data set
regents = regents[(regents.ay == regents.ay.max()) & (regents.dbn.isin(dbns)) & (regents.category == "All Students")]
regents = regents[["dbn", "regents_exam", "number_tested", "mean_score"]]
regents = regents[regents.regents_exam.isin(['Common Core English','Common Core Algebra2'])]
regents.regents_exam = regents.regents_exam.map( {'Common Core English': 'English', 'Common Core Algebra2': 'Algebra2'})
results = regents.pivot(index='dbn', columns='regents_exam', values=['number_tested', 'mean_score']).reset_index()
results.columns = ["dbn", "algebra2_n", "english_n", "algebra2_mean", "english_mean"]
results[["dbn", "algebra2_n", "algebra2_mean", "english_n",  "english_mean"]]
steam = df.merge(results, on='dbn', how='inner')


In [26]:
steam.algebra2_n = steam.algebra2_n.fillna(0).astype(int)
steam.english_n = steam.english_n.fillna(0).astype(int)
data = steam[["dbn","campus_id", "school_name", "total_enrollment", "white_pct", "algebra2_n", "algebra2_mean", "english_n",  "english_mean"]]
data.sort_values("algebra2_mean", ascending=False)


,dbn,campus_id,school_name,total_enrollment,white_pct,algebra2_n,algebra2_mean,english_n,english_mean
8,17K543,569,"Science, Technology and Research Early College...",585,0.029060,21,73.238098,173,78.023125
2,13K595,454,Bedford Academy High School,338,0.047337,41,70.804878,124,84.701614
11,22K405,747,Midwood High School,3642,0.192477,788,69.319794,1832,84.034386
1,13K419,446,"Science Skills Center High School for Science,...",447,0.035794,67,67.865669,306,71.604576
9,17K590,578,Medgar Evers College Preparatory School,1268,0.009464,350,64.077141,314,75.509552
4,13K670,456,Benjamin Banneker Academy,852,0.017606,222,59.540539,308,80.045456
7,17K528,574,The High School for Global Citizenship,214,0.018692,12,53.416668,96,66.437500
10,17K600,574,Clara Barton High School,1324,0.023414,85,52.423531,337,64.738869
3,13K605,455,George Westinghouse Career and Technical Educa...,558,0.034050,65,47.630768,219,65.931503
0,13K265,439,Dr. Susan S. McKinney Secondary School of the ...,130,0.015385,43,42.697674,80,66.087502


In [34]:
feet.to_crs(nyc.crs_meters, inplace=True)
feet["centroid"] = feet.geometry.centroid

x = feet[["campus_id", "centroid"]].merge(data, on='campus_id', how='inner')
x = gpd.GeoDataFrame(x,  crs=feet.crs, geometry='centroid')
x.to_crs(nyc.crs_leaflet, inplace=True)
feet.to_crs(nyc.crs_leaflet, inplace=True)

m = ui.base_map(x, zoom=13, center=steam_coords)

# show these columns in the sidebar popup
cols = ["school_name", "----", "dbn", "total_enrollment", "white_pct", "algebra2_n",
        "algebra2_mean", "english_n",  "english_mean"]

x["sidebar"] = x.apply(ui.popup(cols=cols), axis=1)

# plot the steam center
kw = {"prefix": "fa", "color": "blue", "icon": "cogs"}
icon = folium.Icon(**kw)
folium.Marker(steam_coords, popup="Brooklyn STEAM Center", icon=icon).add_to(m)

# plot all the schools not in the STEAM dataset
m = feet[~feet.campus_id.isin(x.campus_id.unique())].explore(m=m, tooltip=None, popup=None, color="gray")
m = feet[feet.campus_id.isin(x.campus_id.unique())].explore(m=m, tooltip=None, popup=None, color="red")

x.drop_duplicates(inplace=True)

x = ui.radial_cluster(x, "campus_id", r=100)

m = x.explore(m=m, tooltip=["dbn", "school_name"], popup="sidebar", column="dbn", cmap="tab20", legend=False)
m = ui.use_sidebar(m)
# m.save("/home/mxc/Downloads/foomap.html")
m